In [67]:
import datasets
from datasets import Dataset, DatasetDict
# cache_path = '/data/kcl/lpy/hf'
data_path = "ZihanWang314/ragen-datasets"
dataset = datasets.load_dataset(data_path)

In [91]:
from collections import deque
def solve_frozen_lake(game_map_str):
    """
    解析地图并找到所有最短路径。
    """
    # 1. 解析地图字符串为二维列表
    # 去除首尾空白，按行分割
    lines = [line.strip() for line in game_map_str.strip().split('\n')]
    # 按制表符或空格分割每一行
    grid = []
    for line in lines:
        # 处理可能存在的制表符或多个空格
        row = [x for x in line.split() if x]
        grid.append(row)

    rows = len(grid)
    cols = len(grid[0])
    
    start_pos = None
    goal_pos = None
    
    # 2. 找到起点(P)和终点(G)的坐标
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 'P':
                start_pos = (r, c)
            elif grid[r][c] == 'G':
                goal_pos = (r, c)

    if not start_pos or not goal_pos:
        return []

    # 定义动作和对应的坐标变化 (行变, 列变)
    moves = {
        'Up': (-1, 0),
        'Down': (1, 0),
        'Left': (0, -1),
        'Right': (0, 1)
    }

    # 3. BFS 初始化
    # 队列存储: (当前坐标 r, 当前坐标 c, 路径列表)
    queue = deque([(start_pos[0], start_pos[1], [])])
    
    # 记录到达每个格子的最短步数，用于剪枝
    # key: (r, c), value: steps
    min_steps = {start_pos: 0}
    
    shortest_paths = []
    min_path_len = float('inf')

    # 4. 开始搜索
    while queue:
        r, c, path = queue.popleft()

        # 如果当前路径长度已经超过已知最短路径，停止该分支
        if len(path) > min_path_len:
            continue

        # 如果到达终点
        if (r, c) == goal_pos:
            if len(path) < min_path_len:
                # 发现了更短的路径，清空之前的长路径，更新最小值
                min_path_len = len(path)
                shortest_paths = [path]
            elif len(path) == min_path_len:
                # 发现了长度相同的另一条路径，加入结果
                shortest_paths.append(path)
            continue

        # 探索四个方向
        for action, (dr, dc) in moves.items():
            nr, nc = r + dr, c + dc
            
            # 检查边界
            if 0 <= nr < rows and 0 <= nc < cols:
                cell_content = grid[nr][nc]
                
                # 检查是否掉进洞里 ('O')
                if cell_content != 'O':
                    new_dist = len(path) + 1
                    
                    # 关键逻辑：寻找所有最短路径
                    # 如果我们到达该点的步数 <= 之前到达该点的最小步数，则继续探索
                    # (如果是 <，说明发现了新捷径；如果是 ==，说明发现了另一条等长路径)
                    if new_dist <= min_steps.get((nr, nc), float('inf')):
                        min_steps[(nr, nc)] = new_dist
                        queue.append((nr, nc, path + [action]))

    return shortest_paths

def get_reward_model(x):
    data_source, prompt = x
    if data_source == 'frozenlake':
        question = prompt[0]['content'].split('[Cumulative Observations]:\n')[1].split(' \t\nDecide the next action:')[0]
        ground_truth = solve_frozen_lake(question)
        return {'ground_truth':ground_truth}

In [92]:
frozenlake[:10][['data_source', 'prompt']].apply(get_reward_model, axis=1)

0    {'ground_truth': [['Down', 'Down', 'Right'], [...
1                        {'ground_truth': [['Right']]}
2               {'ground_truth': [['Up', 'Up', 'Up']]}
3        {'ground_truth': [['Right', 'Down', 'Down']]}
4    {'ground_truth': [['Up', 'Left', 'Left'], ['Le...
5    {'ground_truth': [['Down', 'Down', 'Down', 'Ri...
6    {'ground_truth': [['Down', 'Left', 'Left'], ['...
7                {'ground_truth': [['Down', 'Right']]}
8    {'ground_truth': [['Right', 'Down', 'Down', 'R...
9    {'ground_truth': [['Up', 'Right'], ['Right', '...
dtype: object

In [ ]:
local_dir = '/home/v-peiyangliu/msranlphot/liupeiyang/dataset/frozenlake'
frozenlake = test_set[test_set['data_source'] == 'frozenlake']
frozenlake = Dataset.from_pandas(frozenlake)
frozenlake.to_parquet(os.path.join(local_dir, "test_verl.parquet"))
frozenlake = train_set[train_set['data_source'] == 'frozenlake']
frozenlake = Dataset.from_pandas(frozenlake)
frozenlake.to_parquet(os.path.join(local_dir, "train_verl.parquet"))

In [72]:
local_dir = '/home/v-peiyangliu/msranlphot/liupeiyang/dataset/sokoban'
test_set = dataset['test'].to_pandas()
sokoban = test_set[test_set['data_source'] == 'sokoban']
sokoban = Dataset.from_pandas(sokoban)
sokoban.to_parquet(os.path.join(local_dir, "test_verl.parquet"))
train_set = dataset['train'].to_pandas()
sokoban = train_set[train_set['data_source'] == 'sokoban']
sokoban = Dataset.from_pandas(sokoban)
sokoban.to_parquet(os.path.join(local_dir, "train_verl.parquet"))

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

61670000

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

8200000